In [1]:
import pandas as pd
import numpy as np

def run_true_realistic_engine(market_data, bankroll=42.00, ego_weight=0.50, kelly_fraction=1.0):
    df = pd.DataFrame(market_data)
    
    eps = 0.001
    pu_clip = np.clip(df['pu'].astype(float), eps, 1-eps)
    pm_clip = np.clip(df['pm'].astype(float), eps, 1-eps)
    
    # 1. Logit Pooling (Bayesian Shrinkage)
    logit_pu = np.log(pu_clip / (1 - pu_clip))
    logit_pm = np.log(pm_clip / (1 - pm_clip))
    logit_norm = (ego_weight * logit_pu) + ((1 - ego_weight) * logit_pm)
    
    df['pu_norm'] = 1 / (1 + np.exp(-logit_norm))
    
    # 2. Action and Pricing
    df['Action'] = np.where(df['pu_norm'] > df['pm'], "YES", "NO")
    df['Price'] = np.where(df['Action'] == "YES", df['pm'], 1 - df['pm'])
    df['Win_Prob'] = np.where(df['Action'] == "YES", df['pu_norm'], 1 - df['pu_norm'])
    
    # 3. Absolute Kelly Calculation
    # We do NOT normalize this. This is the absolute % of your bankroll to risk.
    df['Raw_Kelly_f'] = ((df['Win_Prob'] - df['Price']) / (1 - df['Price'])).clip(lower=0)
    
    # 4. The Half-Kelly Dampener (The Reality Check)
    df['Adjusted_Kelly_f'] = df['Raw_Kelly_f'] * kelly_fraction
    
    # Cap total exposure to 100% of bankroll to prevent margin borrowing
    total_kelly = df['Adjusted_Kelly_f'].sum()
    print(f"Total Kelly Fraction: {total_kelly*100:.2f}%")
    req_bankroll = bankroll * total_kelly
    print(f"Total Bankroll Need: ${req_bankroll:.2f}\n")
    
    if total_kelly > 1.0:
        df['Adjusted_Kelly_f_norm'] = df['Adjusted_Kelly_f'] / total_kelly
        
    # 5. Final Allocation
    df['Allocation'] = df['Adjusted_Kelly_f'] * bankroll
    df['Allocation_norm'] = df['Adjusted_Kelly_f_norm'] * bankroll
    
    return df, req_bankroll

In [2]:
data = {
    "Topic": ["French PE", "Spain", "France", "ICJ", "Spider-Man", "USA ress", 'Dem House', "Hottest year"],
    "pm": [0.22, 0.17, 0.18, 0.02, 0.60, 0.25, 0.81, 0.36], 
    "pu": [0.63, 0.11, 0.13, 0.35, 0.04, 0.23, 0.42, 0.32],
}

bankroll=48
kelly_fraction=1.0

df_real, req_bankroll = run_true_realistic_engine(data, bankroll=bankroll, ego_weight=0.50, kelly_fraction=kelly_fraction)

# Calculate Cash Reserve
total_invested = df_real['Allocation_norm'].sum()
cash_reserve = bankroll - total_invested

print("--- THE TRUE REALISTIC PORTFOLIO ---")
print(f"Total Bankroll: ${bankroll:.2f}")
print(f"Req Bankroll: ${req_bankroll:.2f}")
print(f"Total Deployed: ${total_invested:.2f}")
print(f"Cash Held in Reserve: ${cash_reserve:.2f}\n")
print(df_real[['Topic', 'Action', 'pu_norm', 'Adjusted_Kelly_f', 'Adjusted_Kelly_f_norm', 'Allocation', 'Allocation_norm']].sort_values('Allocation', ascending=False).to_string(index=False))

Total Kelly Fraction: 163.66%
Total Bankroll Need: $78.56

--- THE TRUE REALISTIC PORTFOLIO ---
Total Bankroll: $48.00
Req Bankroll: $78.56
Total Deployed: $48.00
Cash Held in Reserve: $0.00

       Topic Action  pu_norm  Adjusted_Kelly_f  Adjusted_Kelly_f_norm  Allocation  Allocation_norm
  Spider-Man     NO 0.200000          0.666667               0.407342   32.000000        19.552427
   French PE    YES 0.409333          0.242734               0.148314   11.651241         7.119064
   Dem House     NO 0.637289          0.213223               0.130282   10.234705         6.253541
       Spain     NO 0.137266          0.192552               0.117652    9.242488         5.647283
      France     NO 0.153338          0.148119               0.090503    7.109735         4.344143
         ICJ    YES 0.094882          0.076410               0.046688    3.667696         2.241011
Hottest year     NO 0.339714          0.056349               0.034430    2.704770         1.652650
    USA ress    

In [9]:
import requests

response = requests.get("https://gamma-api.polymarket.com/events?closed=false&limit=1000")
events = response.json()

all_tags = set()
for event in events:
    tags = event.get('tags', [])
    for tag in tags:
        if isinstance(tag, dict) and 'label' in tag:
            all_tags.add(tag['label'])

print(sorted(list(all_tags)))

['2025 Predictions', '2026 FIFA World Cup', '2026 NBA Playoffs', '2026 NHL Playoffs', 'AI', 'Abstract', 'Acquisitions', 'Airdrops', 'All', 'Axiom', 'Base', 'Basketball', 'Big Tech', 'Bitcoin', 'Brazil', 'Breaking News', 'Business', 'Celebrities', 'Champions League', 'China', 'Claude 5', 'Colombia', 'Colombia Election', 'Congress', 'Courts', 'Crypto', 'Crypto Prices', 'Culture', 'Databricks', 'Democratic Primary', 'Earn 4%', 'Economy', 'Elections', 'Elon Musk', 'England', 'Epstein', 'FIFA World Cup', 'Fannie Mae', 'Featured', 'Felix', 'Finance', 'Foreign Policy', 'France', 'Freddie Mac', 'GPT-5', 'GTA VI', 'Gaza', 'Geopolitics', 'Ghislaine Maxwell', 'Global', 'Global Elections', 'Grok', 'Grooming Gangs', 'HFC', 'Hide From New', 'Hockey', 'IPO', 'IPOs', 'India', 'Israel', 'Ligue 1', 'Macro Election 2', 'Macro Geopolitics', 'Macron', 'Main Election', 'Maine Primary', 'March 3 Primaries', 'Maxwell', 'MegaETH', 'Metamask', 'Michigan Primary', 'MicroStrategy', 'Middle East', 'Midterms', 'Mil

In [ ]:
import requests
import json
import os
import pandas as pd
import numpy as np

# --- 1. Your Existing Math Engine ---
def run_true_realistic_engine(market_data, bankroll=42.00, ego_weight=0.50, kelly_fraction=1.0):
    df = pd.DataFrame(market_data)
    if df.empty: return df, 0
    
    eps = 0.001
    pu_clip = np.clip(df['pu'].astype(float), eps, 1-eps)
    pm_clip = np.clip(df['pm'].astype(float), eps, 1-eps)
    
    logit_pu = np.log(pu_clip / (1 - pu_clip))
    logit_pm = np.log(pm_clip / (1 - pm_clip))
    logit_norm = (ego_weight * logit_pu) + ((1 - ego_weight) * logit_pm)
    
    df['pu_norm'] = 1 / (1 + np.exp(-logit_norm))
    df['Action'] = np.where(df['pu_norm'] > df['pm'], "YES", "NO")
    df['Price'] = np.where(df['Action'] == "YES", df['pm'], 1 - df['pm'])
    df['Win_Prob'] = np.where(df['Action'] == "YES", df['pu_norm'], 1 - df['pu_norm'])
    
    df['Raw_Kelly_f'] = ((df['Win_Prob'] - df['Price']) / (1 - df['Price'])).clip(lower=0)
    df['Adjusted_Kelly_f'] = df['Raw_Kelly_f'] * kelly_fraction
    
    total_kelly = df['Adjusted_Kelly_f'].sum()
    req_bankroll = bankroll * total_kelly
    
    df['Adjusted_Kelly_f_norm'] = df['Adjusted_Kelly_f'] / total_kelly if total_kelly > 1.0 else df['Adjusted_Kelly_f']
    
    df['Allocation'] = df['Adjusted_Kelly_f'] * bankroll
    df['Allocation_norm'] = df['Adjusted_Kelly_f_norm'] * bankroll
    
    return df, req_bankroll

# --- 2. The Discovery & Tracking Wrapper ---
HISTORY_FILE = "prediction_history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            return json.load(f)
    return {"seen_ids": []}

def save_history(history):
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f)

def get_blind_predictions(category="Politics", limit=3):
    history = load_history()
    seen_ids = set(history["seen_ids"])
    
    url = "https://gamma-api.polymarket.com/events?active=true&closed=false&limit=100"
    events = requests.get(url).json()
    
    market_data = []
    presented_count = 0
    
    print(f"\n--- BLIND PREDICTION PHASE: {category.upper()} ---")
    print("Evaluate the probabilities without seeing the market odds.\n")
    
    for event in events:
        if presented_count >= limit: break
        
        event_id = event.get('id')
        if event_id in seen_ids: continue
            
        tags = [t.get('label', '').lower() for t in event.get('tags', []) if isinstance(t, dict)]
        if category.lower() not in tags: continue
            
        markets = [m for m in event.get('markets', []) if m.get('active')]
        if not markets: continue
        
        # Mark as seen so we don't present it again tomorrow
        history["seen_ids"].append(event_id)
        
        market = markets[0] # Grab the primary market question
        title = event.get('title')
        question = market.get('question')
        pm = float(market.get('outcomePrices', ['0.5'])[0]) # Current YES price
        
        print(f"Event: {title}")
        print(f"Question: {question}")
        user_input = input("Your Probability % (0-100), or 's' to skip: ")
        
        if user_input.lower() == 's':
            print("Skipped.\n")
            continue
            
        try:
            pu = float(user_input) / 100.0
            market_data.append({
                "Topic": title[:15] + "...", # Shortened for DataFrame viewing
                "pm": pm,
                "pu": pu
            })
            print(f"> Logged! (Market was actually: {pm*100:.1f}%)\n")
        except ValueError:
            print("> Invalid input, skipped.\n")
            
        presented_count += 1
        
    save_history(history)
    return market_data

# --- 3. Execution Pipeline ---
if __name__ == "__main__":
    my_predictions = get_blind_predictions(category="Politics", limit=3)
    
    if my_predictions:
        df_real, req_bank = run_true_realistic_engine(my_predictions, bankroll=48, ego_weight=0.5, kelly_fraction=0.5)
        print("\n--- FINAL PORTFOLIO ALLOCATION ---")
        print(df_real[['Topic', 'pm', 'pu', 'Action', 'Allocation_norm']].to_string(index=False))
    else:
        print("No predictions made.")

In [10]:
import pandas as pd
import numpy as np

def run_volume_adjusted_engine(market_data, bankroll=5000.00, ego_weight=0.50, kelly_fraction=0.5, max_volume_impact=0.01):
    """
    max_volume_impact (alpha): The maximum percentage of a market's total volume 
                               you are willing to represent. e.g., 0.01 = 1%.
    """
    df = pd.DataFrame(market_data)
    eps = 0.001
    
    pu_clip = np.clip(df['pu'].astype(float), eps, 1-eps)
    pm_clip = np.clip(df['pm'].astype(float), eps, 1-eps)
    
    # 1. Bayesian Shrinkage
    logit_pu = np.log(pu_clip / (1 - pu_clip))
    logit_pm = np.log(pm_clip / (1 - pm_clip))
    logit_norm = (ego_weight * logit_pu) + ((1 - ego_weight) * logit_pm)
    df['pu_norm'] = 1 / (1 + np.exp(-logit_norm))
    
    # 2. Action & Pricing
    df['Action'] = np.where(df['pu_norm'] > df['pm'], "YES", "NO")
    df['Price'] = np.where(df['Action'] == "YES", df['pm'], 1 - df['pm'])
    df['Win_Prob'] = np.where(df['Action'] == "YES", df['pu_norm'], 1 - df['pu_norm'])
    
    # 3. Base Kelly
    df['Raw_Kelly_f'] = ((df['Win_Prob'] - df['Price']) / (1 - df['Price'])).clip(lower=0)
    df['Theoretical_Allocation'] = df['Raw_Kelly_f'] * kelly_fraction * bankroll
    
    # 4. SLIPPAGE PROTECTION (The Volume Cap)
    # Ensure our allocation never exceeds a safe fraction of the market's total volume
    df['Max_Safe_Bet'] = df['volume'] * max_volume_impact
    
    # The actual allocation is the lesser of the Kelly recommendation or the Liquidity Cap
    df['Final_Allocation'] = np.minimum(df['Theoretical_Allocation'], df['Max_Safe_Bet'])
    
    return df

# --- Example Data with Volume ---
data = {
    "Topic": ["French PE", "Obscure Local Race"],
    "pm": [0.22, 0.55], 
    "pu": [0.63, 0.85],
    "volume": [67000000, 4500] # French PE has $67M volume, local race has $4.5k
}

df_safe = run_volume_adjusted_engine(data, bankroll=10000)
print(df_safe[['Topic', 'Theoretical_Allocation', 'Max_Safe_Bet', 'Final_Allocation']])

                Topic  Theoretical_Allocation  Max_Safe_Bet  Final_Allocation
0           French PE             1213.670940      670000.0        1213.67094
1  Obscure Local Race             1940.533632          45.0          45.00000


In [11]:
# The fee parameters exactly as specified in the Polymarket Documentation
FEE_RATES = {
    "Crypto": 0.07,
    "Sports": 0.03,
    "Finance": 0.04,
    "Politics": 0.04,
    "Economics": 0.05,
    "Culture": 0.05,
    "Weather": 0.05,
    "Mentions": 0.04,
    "Tech": 0.04,
    "Geopolitics": 0.00,
    "Other / General": 0.05
}

def map_to_fee_category(event_tags):
    """
    Evaluates raw API tags and assigns the event to the strictest 
    matching Polymarket fee category.
    """
    tags_lower = [str(t).lower() for t in event_tags]
    
    # Mapping logic ordered by priority (e.g., Geopolitics beats general Politics)
    category_mapping = {
        "Geopolitics": ["geopolitics", "middle east", "israel", "ukraine", "russia", "world affairs", "syria", "gaza", "nato"],
        "Politics": ["politics", "elections", "us election", "trump", "biden", "dem house", "senate", "primaries", "congress"],
        "Crypto": ["crypto", "bitcoin", "ethereum", "stablecoins", "token launch", "airdrops", "usdt", "base", "megaeth"],
        "Sports": ["sports", "nba", "nhl", "fifa", "champions league", "hockey", "soccer", "basketball", "ucl"],
        "Tech": ["tech", "ai", "openai", "gpt-5", "tiktok", "elon musk", "big tech", "claude 5", "space x"],
        "Finance": ["finance", "stocks", "ipo", "interest rates", "fed", "acquisitions", "microstrategy"],
        "Economics": ["economy", "macro", "taxes", "inflation"],
        "Culture": ["culture", "movies", "taylor swift", "travis kelce", "celebrities", "gta vi", "music", "epstein"],
        "Weather": ["weather", "hottest year", "climate"],
        "Mentions": ["mentions", "tweets", "x posts"]
    }
    
    for fee_category, keywords in category_mapping.items():
        if any(kw in tags_lower for kw in keywords):
            return fee_category
            
    return "Other / General"

In [25]:
import pandas as pd
import numpy as np

def run_fee_adjusted_engine(market_data, bankroll=48.00, ego_weight=0.50, kelly_fraction=1.0):
    df = pd.DataFrame(market_data)
    eps = 0.001
    
    pu_clip = np.clip(df['pu'].astype(float), eps, 1-eps)
    pm_clip = np.clip(df['pm'].astype(float), eps, 1-eps)
    
    # 1. Logit Pooling (Bayesian Shrinkage)
    logit_pu = np.log(pu_clip / (1 - pu_clip))
    logit_pm = np.log(pm_clip / (1 - pm_clip))
    logit_norm = (ego_weight * logit_pu) + ((1 - ego_weight) * logit_pm)
    df['pu_norm'] = 1 / (1 + np.exp(-logit_norm))
    
    # 2. Action and Base Pricing
    df['Action'] = np.where(df['pu_norm'] > df['pm'], "YES", "NO")
    df['Base_Price'] = np.where(df['Action'] == "YES", df['pm'], 1 - df['pm'])
    df['Win_Prob'] = np.where(df['Action'] == "YES", df['pu_norm'], 1 - df['pu_norm'])
    
    # 3. POLYMARKET DYNAMIC FEE CALCULATION
    # Fee per share = feeRate * p * (1 - p)
    df['Fee_Rate'] = df['Category'].map(FEE_RATES).fillna(0.05)
    df['Fee_Per_Share'] = df['Fee_Rate'] * df['Base_Price'] * (1 - df['Base_Price'])
    
    # The true cost to acquire the asset
    df['True_Price'] = df['Base_Price'] + df['Fee_Per_Share']
    
    # 4. Fee-Adjusted Kelly Calculation
    # We replace 'Price' with 'True_Price' to account for the mathematical drag of the fee
    df['Raw_Kelly_f'] = ((df['Win_Prob'] - df['True_Price']) / (1 - df['True_Price'])).clip(lower=0)
    df['Adjusted_Kelly_f'] = df['Raw_Kelly_f'] * kelly_fraction
    
    total_kelly = df['Adjusted_Kelly_f'].sum()
    req_bankroll = bankroll * total_kelly
    
    if total_kelly > 1.0:
        df['Adjusted_Kelly_f_norm'] = df['Adjusted_Kelly_f'] / total_kelly
    else:
        df['Adjusted_Kelly_f_norm'] = df['Adjusted_Kelly_f']
        
    df['Allocation'] = df['Adjusted_Kelly_f_norm'] * bankroll
    
    return df, req_bankroll

# ==========================================
# Testing the Engine with Fee Discrepancies
# ==========================================
data = {
    "Topic": ["Iran Peace Deal", "BTC Up or Down 5m", "Texas Senate"],
    "Category": ["Geopolitics", "Crypto", "Politics"],
    "pm": [0.50, 0.50, 0.50], # All have the exact same market price
    "pu": [0.60, 0.60, 0.60]  # You have the exact same conviction for all
}

df_fees, req_bankroll = run_fee_adjusted_engine(data, bankroll=48)

print("--- FEE ADJUSTED ALLOCATIONS ---")
print(df_fees[['Topic', 'Category', 'Base_Price', 'True_Price', 'Allocation']].to_string(index=False))

--- FEE ADJUSTED ALLOCATIONS ---
            Topic    Category  Base_Price  True_Price  Allocation
  Iran Peace Deal Geopolitics         0.5      0.5000    4.848985
BTC Up or Down 5m      Crypto         0.5      0.5175    3.283922
     Texas Senate    Politics         0.5      0.5100    3.968352


# Polymarket API

## debug

- bid and ask price deadzone bets 
- random market shuffle
- volume impact calcuation - manually set to 1% (may downscale allocation when confident)
- bet details

In [ ]:
import requests
import json
import os
import random
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# --- CONFIGURATION ---
HISTORY_FILE = "prediction_history.json"
BANKROLL = 420000.00
EGO_WEIGHT = 0.50
KELLY_FRACTION = 1.0 
MAX_VOLUME_IMPACT = 0.01 

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f: 
            return json.load(f)
    return {"seen_market_ids": []}

def save_history(history):
    with open(HISTORY_FILE, 'w') as f: 
        json.dump(history, f)

def calculate_allocation(pu, pm_bid, pm_ask, fee_rate, volume, bankroll, ego, kelly, max_vol):
    """Calculates allocation factoring in Bayesian shrinkage, fees, and Bid-Ask spread."""
    eps = 0.001
    pu_clip = np.clip(pu, eps, 1-eps)
    
    # We pool our belief against the market's 'Mid Price' consensus
    pm_mid = (pm_bid + pm_ask) / 2.0
    pm_mid_clip = np.clip(pm_mid, eps, 1-eps)
    
    # 1. Bayesian Shrinkage (Logit Pooling)
    logit_pu, logit_pm = np.log(pu_clip / (1-pu_clip)), np.log(pm_mid_clip / (1-pm_mid_clip))
    logit_norm = (ego * logit_pu) + ((1 - ego) * logit_pm)
    pu_norm = 1 / (1 + np.exp(-logit_norm)) 
    
    # 2. Action & Execution Pricing (Crossing the Spread)
    cost_yes = pm_ask
    cost_no = 1.0 - pm_bid # Buying NO means taking the opposite of the YES Bid
    
    if pu_norm > cost_yes:
        action = "YES"
        base_price = cost_yes
        win_prob = pu_norm
    elif (1.0 - pu_norm) > cost_no:
        action = "NO"
        base_price = cost_no
        win_prob = 1.0 - pu_norm
    else:
        # Edge is too small to overcome the bid-ask spread
        return "NONE", 0.0, 0.0, 0.0
    
    # 3. Dynamic Fee Adjustment
    true_price = base_price + (fee_rate * base_price * (1 - base_price))
    
    # 4. Fractional Kelly Formula
    raw_kelly = max(0, (win_prob - true_price) / (1 - true_price))
    adj_kelly = raw_kelly * kelly
    
    # 5. Volume Caps (Slippage Protection)
    theoretical_allocation = adj_kelly * bankroll
    liquidity_cap = volume * max_vol
    final_allocation = min(theoretical_allocation, liquidity_cap)
    
    return action, true_price, adj_kelly, final_allocation

def run_prediction_session():
    history = load_history()
    seen_ids = set(history.get("seen_market_ids", []))
    
    url = "https://gamma-api.polymarket.com/events?active=true&closed=false&limit=300"
    try:
        events = requests.get(url).json()
    except Exception as e:
        print(f"Error fetching data from API: {e}")
        return pd.DataFrame()
        
    # FIX 1: Extract all active sub-markets into a flat list for global shuffling
    all_active_markets = []
    for event in events:
        event_slug = event.get('slug', '')
        for m in event.get('markets', []):
            if not m.get('active') or m.get('closed') or m.get('umaResolutionStatus') == 'resolved':
                continue
            # Append the parent URL slug so we can link to it later
            m['parent_slug'] = event_slug
            all_active_markets.append(m)

    # Apply global randomization across all markets and categories
    random.shuffle(all_active_markets) 
    
    portfolio_data = []
    cumulative_exposure = 0.0 
    
    print("\n--- Starting Prediction Session ---")
    print(f"Bankroll: ${BANKROLL:,.2f} | Max Exposure: 100% (1.0 Kelly sum)\n")
    
    for m in all_active_markets:
        if cumulative_exposure >= 1.0:
            print("\nMaximum capital exposure reached (100%). Session ending.")
            break
            
        market_id = m.get('id')
        if market_id in seen_ids: 
            continue
            
        # Parse exact pricing and order book spread
        raw_prices_str = m.get('outcomePrices', '["0.5", "0.5"]')
        try:
            pm_mid = float(json.loads(raw_prices_str)[0])
        except (json.JSONDecodeError, ValueError, IndexError):
            pm_mid = 0.50
            
        # FIX 2: Safely extract Bid and Ask, using Mid Price as fallback if order book is empty
        try:
            pm_bid = float(m.get('bestBid', pm_mid))
            pm_ask = float(m.get('bestAsk', pm_mid))
        except (TypeError, ValueError):
            pm_bid = pm_mid
            pm_ask = pm_mid
            
        volume = float(m.get('volumeNum', 0))
        fee_rate = m.get('feeSchedule', {}).get('rate', 0.05)
        spread = float(m.get('spread', 0))
        question = m.get('question', 'Unknown Question')
        event_url = f"https://polymarket.com/event/{m.get('parent_slug')}"
        
        seen_ids.add(market_id)
        history["seen_market_ids"].append(market_id)
        
        print(f"Market: {question}")
        user_input = input("Enter your probability % (0-100), or 's' to skip: ")
        
        if user_input.lower() == 's': 
            print("Skipped.\n")            
            continue

        # Calculate days until resolution
        end_date_str = m.get('endDate')
        try:
            resolution_dt = datetime.fromisoformat(end_date_str.replace('Z', '+00:00'))
            now = datetime.now(timezone.utc)
            days_until = (resolution_dt - now).days
            days_str = f"{days_until} days remaining"
        except:
            days_str = "Resolution date unknown"

        # Print pm and pu for reference
        print(f"Your Input: {float(user_input):.1f}% | Market Mid Price: {pm_mid*100:.1f}%")
        # Provide the full market details
        print(f"Context: Spread {spread*100:.1f}% | Total Volume: ${volume:,.0f}")
        print(f"(Spread details: Bid {pm_bid*100:.1f}% | Ask {pm_ask*100:.1f}%)")
        print(f"Days until resolution: {days_str}")
        print(f"Fee Rate: {fee_rate*100:.1f}%\n")

        try:
            pu = float(user_input) / 100.0
            
            action, true_price, adj_kelly, final_alloc = calculate_allocation(
                pu, pm_bid, pm_ask, fee_rate, volume, BANKROLL, EGO_WEIGHT, KELLY_FRACTION, MAX_VOLUME_IMPACT
            )
            
            if final_alloc > 0:
                cumulative_exposure += adj_kelly
                print(f"Action: Buy {action} @ {true_price*100:.1f}% (Includes Spread & Fee)")
                print(f"Allocation: ${final_alloc:,.2f}")
                print(f"Link: {event_url}")
                print(f"Current Total Exposure: {cumulative_exposure*100:.1f}%\n")
                

                portfolio_data.append({
                    "Question": question,
                    "Action": action,
                    "True_Price": f"{true_price*100:.1f}%",
                    "Allocation": f"${final_alloc:,.2f}",
                    "Link": event_url
                })
            else:
                if action == "NONE":
                    print("Result: $0 allocation (Edge is trapped inside the bid-ask spread).")
                else:
                    print("Result: $0 allocation (No mathematical edge against fee/odds).")
                print(f"Link: {event_url}\n")
                
        except ValueError: 
            print("Invalid input format. Skipped.\n")
            
    save_history(history)
    return pd.DataFrame(portfolio_data)

# --- EXECUTION ---
if __name__ == "__main__":
    portfolio = run_prediction_session()
    
    if not portfolio.empty:
        print("\n--- Final Session Allocations ---")
        pd.set_option('display.max_colwidth', None) 
        print(portfolio.to_string(index=False))
    else:
        print("\nNo allocations were made during this session.")


--- Starting Prediction Session ---
Bankroll: $420,000.00 | Max Exposure: 100% (1.0 Kelly sum)

Market: Will GPT-6 be released by September 30, 2026?
Your Input: 0.0% | Market Mid Price: 52.0%
Context: Spread 6.0% | Total Volume: $6,155
(Spread details: Bid 49.0% | Ask 55.0%)
Days until resolution: 33 days remaining
Fee Rate: 4.0%

Action: Buy NO @ 52.0% (Includes Spread & Fee)
Allocation: $61.55
Link: https://polymarket.com/event/gpt-6-released-by
Current Total Exposure: 93.4%

Market: Will Viktor Gyokeres be the 2025/2026 top UCL goal scorer?
Your Input: 0.0% | Market Mid Price: 0.1%
Context: Spread 0.1% | Total Volume: $48,376
(Spread details: Bid 0.1% | Ask 0.1%)
Days until resolution: 2 days remaining
Fee Rate: 3.0%

Result: $0 allocation (Edge is trapped inside the bid-ask spread).
Link: https://polymarket.com/event/champions-league-top-scorer-655

Market: Will Ecuador win the 2026 FIFA World Cup?
Your Input: 0.0% | Market Mid Price: 0.8%
Context: Spread 0.1% | Total Volume: $27

## above errors (fixed below)
- negative days until resolution market proposed 
- total exposure when volume impact and allocation error, still adds the normal exposure % without volume reductions and allocation stops when I still have capital 

- prediction bounds and dynamic ego (0-0.5 base)
- Review markets (repredict)

In [ ]:
import requests
import json
import os
import random
import re
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
HISTORY_FILE = "prediction_history.json"
BANKROLL = 42.00
BASE_EGO_WEIGHT = 0.50 
KELLY_FRACTION = 1.0
MAX_VOLUME_IMPACT = 0.01 

def load_history():
    """Loads history, migrating old formats gracefully to prevent crashes."""
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, 'r') as f: 
                data = json.load(f)
                # Failsafe: If the old JSON is a list or uses old keys, reset gracefully
                if isinstance(data, list) or "seen_market_ids" in data:
                    return {"predicted": {}, "skipped": {}}
                return data
        except json.JSONDecodeError:
            pass
    return {"predicted": {}, "skipped": {}}

def save_history(history):
    with open(HISTORY_FILE, 'w') as f: 
        json.dump(history, f, indent=4)

def parse_user_input(user_input):
    """Parses bounds '40-60' or exact '50' into floats."""
    numbers = [float(x) for x in re.findall(r'\d+\.?\d*', user_input)]
    if len(numbers) == 1:
        return numbers[0] / 100.0, numbers[0] / 100.0
    elif len(numbers) >= 2:
        return min(numbers[0], numbers[1]) / 100.0, max(numbers[0], numbers[1]) / 100.0
    else:
        raise ValueError("No numbers found")

def calculate_allocation(lower_bound, upper_bound, pm_bid, pm_ask, fee_rate, volume, bankroll, base_ego, kelly, max_vol):
    """Calculates Dynamic Ego, Logit Pooling, and Fee-Adjusted Kelly."""
    eps = 0.001
    
    pu_mid = (lower_bound + upper_bound) / 2.0
    spread = upper_bound - lower_bound
    
    # Dynamic Ego reduces smoothly as uncertainty widens
    dynamic_ego = max(0.00, base_ego * (1.0 - spread))
    
    pu_clip = np.clip(pu_mid, eps, 1-eps)
    pm_mid = (pm_bid + pm_ask) / 2.0
    pm_mid_clip = np.clip(pm_mid, eps, 1-eps)
    
    # Bayesian Shrinkage
    logit_pu, logit_pm = np.log(pu_clip / (1-pu_clip)), np.log(pm_mid_clip / (1-pm_mid_clip))
    logit_norm = (dynamic_ego * logit_pu) + ((1 - dynamic_ego) * logit_pm)
    pu_norm = 1 / (1 + np.exp(-logit_norm)) 
    
    cost_yes = pm_ask
    cost_no = 1.0 - pm_bid 
    
    if pu_norm > cost_yes:
        action, base_price, win_prob = "YES", cost_yes, pu_norm
    elif (1.0 - pu_norm) > cost_no:
        action, base_price, win_prob = "NO", cost_no, 1.0 - pu_norm
    else:
        return "NONE", 0.0, 0.0, 0.0, dynamic_ego
    
    true_price = base_price + (fee_rate * base_price * (1 - base_price))
    raw_kelly = max(0, (win_prob - true_price) / (1 - true_price))
    adj_kelly = raw_kelly * kelly
    
    final_allocation = min(adj_kelly * bankroll, volume * max_vol)
    return action, true_price, adj_kelly, final_allocation, dynamic_ego

def run_prediction_session(mode="discover"):
    history = load_history()
    
    print("\nFetching active market data...")
    try:
        events = requests.get("https://gamma-api.polymarket.com/events?active=true&closed=false&limit=400").json()
    except Exception as e:
        print(f"API Error: {e}")
        return pd.DataFrame()
        
    all_active_markets = []
    for event in events:
        for m in event.get('markets', []):
            if not m.get('active') or m.get('closed') or m.get('umaResolutionStatus') == 'resolved':
                continue
                
            market_id = m.get('id')
            if mode == "discover" and (market_id in history["predicted"] or market_id in history["skipped"]):
                continue
            if mode == "review" and (market_id not in history["predicted"] and market_id not in history["skipped"]):
                continue
                
            m['parent_slug'] = event.get('slug', '')
            all_active_markets.append(m)

    random.shuffle(all_active_markets) 
    
    if not all_active_markets:
        print("\nNo valid markets found for the selected mode.")
        return pd.DataFrame()

    portfolio_data = []
    cumulative_exposure = 0.0 
    
    print(f"\n--- Starting Session [{mode.upper()} MODE] ---")
    print(f"Bankroll: ${BANKROLL:,.2f} | Max Exposure: 100%\n")
    
    for m in all_active_markets:
        if cumulative_exposure >= 1.0:
            print("\n[!] Maximum capital exposure reached (100%). Session ending.")
            break
            
        market_id = m.get('id')
        question = m.get('question', 'Unknown Question')
        event_url = f"https://polymarket.com/event/{m.get('parent_slug')}"
        today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
        
        # 1. Blind Prompt (User sees ONLY the question to prevent anchoring)
        print(f"==================================================")
        print(f"Market: {question}")
        user_input = input("Enter bounds ('40-60'), exact % ('50'), or 's' to skip: ")
        
        if user_input.lower() == 's': 
            print("Status: Skipped.\n")
            history["skipped"][market_id] = {"date": today_str}
            history["predicted"].pop(market_id, None) 
            continue
        
        try:
            # 2. Parse Market Data (Calculated strictly behind the scenes)
            lower, upper = parse_user_input(user_input)
            
            try: pm_mid = float(json.loads(m.get('outcomePrices', '["0.5"]'))[0])
            except: pm_mid = 0.50
            try: pm_bid, pm_ask = float(m.get('bestBid', pm_mid)), float(m.get('bestAsk', pm_mid))
            except: pm_bid, pm_ask = pm_mid, pm_mid
                
            volume = float(m.get('volumeNum', 0))
            fee_rate = m.get('feeSchedule', {}).get('rate', 0.05)
            
            try:
                res_dt = datetime.fromisoformat(m.get('endDate').replace('Z', '+00:00'))
                days_str = f"{(res_dt - datetime.now(timezone.utc)).days} Days"
            except:
                days_str = "Unknown"

            # 3. Math Engine
            action, true_price, adj_kelly, final_alloc, dynamic_ego = calculate_allocation(
                lower, upper, pm_bid, pm_ask, fee_rate, volume, BANKROLL, BASE_EGO_WEIGHT, KELLY_FRACTION, MAX_VOLUME_IMPACT
            )
            
            # 4. Save Deep Context to History
            history["predicted"][market_id] = {
                "date": today_str,
                "bounds": f"{(lower*100):.0f}% - {(upper*100):.0f}%",
                "allocation": round(final_alloc, 2)
            }
            history["skipped"].pop(market_id, None)
            
            # 5. Full Post-Prediction Market Profile Reveal
            print(f"\n--- MARKET ANALYSIS ---")
            print(f"Volume       : ${volume:,.0f}")
            print(f"Resolution   : {days_str}")
            print(f"Fee Rate     : {fee_rate*100:.1f}%")
            print(f"Market Spread: Bid {pm_bid*100:.1f}% | Ask {pm_ask*100:.1f}%")
            print(f"User Bounds  : {lower*100:.1f}% to {upper*100:.1f}% (Spread: {(upper-lower)*100:.1f}%)")
            print(f"Dynamic Ego  : {dynamic_ego:.2f} (Base: {BASE_EGO_WEIGHT})")
            print(f"-----------------------")
            
            if final_alloc > 0:
                cumulative_exposure += adj_kelly
                print(f"ACTION       : BUY {action} @ {true_price*100:.1f}%")
                print(f"ALLOCATION   : ${final_alloc:,.2f}")
                print(f"EXPOSURE     : {cumulative_exposure*100:.1f}%")
                print(f"LINK         : {event_url}\n")
                
                portfolio_data.append({
                    "Question": question[:50] + "..",
                    "Action": action,
                    "Ego": f"{dynamic_ego:.2f}",
                    "Price": f"{true_price*100:.1f}%",
                    "Alloc": f"${final_alloc:,.0f}"
                })
            else:
                reason = "Trapped in bid-ask spread" if action == "NONE" else "No mathematical edge"
                print(f"ACTION       : $0 Allocation ({reason})")
                print(f"LINK         : {event_url}\n")
                
        except ValueError: 
            print("Error: Invalid input format. Skipped.\n")
            
    save_history(history)
    return pd.DataFrame(portfolio_data)

if __name__ == "__main__":
    print("Select Mode:\n 1: Discover New Markets\n 2: Review Previous Markets")
    choice = input("> ")
    op_mode = "review" if choice.strip() == "2" else "discover"
    
    portfolio = run_prediction_session(mode=op_mode)
    
    if not portfolio.empty:
        print("\n--- Final Session Allocations ---")
        pd.set_option('display.max_colwidth', None) 
        print(portfolio.to_string(index=False))
    else:
        print("\nNo allocations made.")

Select Mode:
 1: Discover New Markets
 2: Review Previous Markets



Fetching active market data...

--- Starting Session [REVIEW MODE] ---
Bankroll: $42.00 | Max Exposure: 100%

Market: Will Eric Trump win the 2028 Republican presidential nomination?

--- MARKET ANALYSIS ---
Volume       : $8,853,274
Resolution   : 894 Days
Fee Rate     : 4.0%
Market Spread: Bid 0.7% | Ask 0.8%
User Bounds  : 0.0% to 0.0% (Spread: 0.0%)
Dynamic Ego  : 0.50 (Base: 0.5)
-----------------------
ACTION       : BUY NO @ 99.3%
ALLOCATION   : $24.86
EXPOSURE     : 59.2%
LINK         : https://polymarket.com/event/republican-presidential-nominee-2028

Market: Mike Johnson out as Speaker by December 31?

--- MARKET ANALYSIS ---
Volume       : $2,798
Resolution   : 218 Days
Fee Rate     : 4.0%
Market Spread: Bid 10.0% | Ask 16.0%
User Bounds  : 0.0% to 0.0% (Spread: 0.0%)
Dynamic Ego  : 0.50 (Base: 0.5)
-----------------------
ACTION       : BUY NO @ 90.4%
ALLOCATION   : $27.98
EXPOSURE     : 146.7%
LINK         : https://polymarket.com/event/mike-johnson-out-as-speaker-by


--

## test codes above and below
### above errors
- when reviewing the "predicted" markets, they should still be in "predicted" if we decide to skip it and not moved to "skipped" when saving to file

## Main Code

- adapted base ego from brier score with historical user and market prediction accuracy 
- add resolved markets to history

In [48]:
import requests
import json
import os
import random
import re
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
HISTORY_FILE = "prediction_history.json"
BANKROLL = 42.00
KELLY_FRACTION = 1.0
MAX_VOLUME_IMPACT = 0.01 

# --- MICROSTRUCTURE DEFENSES ---
MIN_EDGE = 0.02          # 2% minimum mathematical edge to bother executing
MAX_DAYS = 90            # Ignore markets locking up capital for more than 3 months
MAX_SPREAD = 0.15        # Ignore markets with bid-ask spreads wider than 15%
EXTREME_ODDS = 0.03      # Ignore tail-risk markets below 3% or above 97%

def load_history():
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, 'r') as f: 
                data = json.load(f)
                if isinstance(data, list) or "seen_market_ids" in data:
                    return {"predicted": {}, "skipped": {}, "resolved": {}}
                if "resolved" not in data:
                    data["resolved"] = {}
                return data
        except json.JSONDecodeError:
            pass
    return {"predicted": {}, "skipped": {}, "resolved": {}}

def save_history(history):
    with open(HISTORY_FILE, 'w') as f: 
        json.dump(history, f, indent=4)

def update_resolutions(history):
    resolved_count = 0
    for market_id in list(history.get("predicted", {}).keys()):
        try:
            resp = requests.get(f"https://gamma-api.polymarket.com/markets/{market_id}").json()
            if resp.get("closed") and not resp.get("active"):
                prices = json.loads(resp.get("outcomePrices", '["0.5", "0.5"]'))
                if prices[0] in ["1", "1.0"]: outcome = 1.0
                elif prices[1] in ["1", "1.0"]: outcome = 0.0
                else: continue 
                
                pred_data = history["predicted"][market_id]
                history["resolved"][market_id] = {
                    "question": pred_data.get("question", "Unknown"),
                    "pu": pred_data.get("pu", 0.5),
                    "pm": pred_data.get("pm", 0.5),
                    "outcome": outcome,
                    "date_resolved": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
                }
                del history["predicted"][market_id]
                resolved_count += 1
        except Exception:
            continue
            
    if resolved_count > 0:
        print(f"\n[*] BACKGROUND SYSTEM: Auto-resolved and scored {resolved_count} closed markets.")
    return history

def calculate_base_ego(history):
    resolved = history.get("resolved", {})
    if not resolved:
        return 0.50 
        
    bs_u_total, bs_m_total = 0.0, 0.0
    
    for data in resolved.values():
        outcome = data.get("outcome", 0)
        bs_u_total += (data.get("pu", 0.5) - outcome) ** 2
        bs_m_total += (data.get("pm", 0.5) - outcome) ** 2
        
    if bs_u_total == 0 and bs_m_total == 0: 
        return 0.50 
        
    return bs_m_total / (bs_u_total + bs_m_total)

def parse_user_input(user_input):
    numbers = [float(x) for x in re.findall(r'\d+\.?\d*', user_input)]
    if len(numbers) == 1:
        return numbers[0] / 100.0, numbers[0] / 100.0
    elif len(numbers) >= 2:
        return min(numbers[0], numbers[1]) / 100.0, max(numbers[0], numbers[1]) / 100.0
    else:
        raise ValueError("No numbers found")

def calculate_allocation(lower_bound, upper_bound, pm_bid, pm_ask, fee_rate, volume, bankroll, base_ego, kelly, max_vol):
    eps = 0.01
    pu_mid = (lower_bound + upper_bound) / 2.0
    spread = upper_bound - lower_bound
    dynamic_ego = max(0.00, base_ego * (1.0 - spread))
    
    pu_clip, pm_mid_clip = np.clip(pu_mid, eps, 1-eps), np.clip((pm_bid + pm_ask) / 2.0, eps, 1-eps)
    
    logit_pu, logit_pm = np.log(pu_clip / (1-pu_clip)), np.log(pm_mid_clip / (1-pm_mid_clip))
    logit_norm = (dynamic_ego * logit_pu) + ((1 - dynamic_ego) * logit_pm)
    pu_norm = 1 / (1 + np.exp(-logit_norm)) 
    
    cost_yes, cost_no = pm_ask, 1.0 - pm_bid 
    
    if pu_norm > cost_yes:
        action, base_price, win_prob = "YES", cost_yes, pu_norm
    elif (1.0 - pu_norm) > cost_no:
        action, base_price, win_prob = "NO", cost_no, 1.0 - pu_norm
    else:
        return "NONE", 0.0, 0.0, 0.0, dynamic_ego, 0.0
    
    true_price = base_price + (fee_rate * base_price * (1 - base_price))
    
    # Explicitly calculate the mathematical edge
    edge = win_prob - true_price
    
    # MICROSTRUCTURE: Min Edge Filter uses actual Edge
    if edge < MIN_EDGE:
        return "THIN_EDGE", true_price, 0.0, 0.0, dynamic_ego, edge
    
    # Kelly fraction based on the edge
    raw_kelly = max(0, edge / (1 - true_price))
    adj_kelly = raw_kelly * kelly
    
    final_allocation = min(adj_kelly * bankroll, volume * max_vol)
    return action, true_price, adj_kelly, final_allocation, dynamic_ego, edge

def run_prediction_session(mode="discover", sub_mode="all", target_slugs=None):
    history = update_resolutions(load_history())
    live_base_ego = calculate_base_ego(history)
    
    print(f"\n--- Starting Session [{mode.upper()} - {sub_mode.upper()}] ---")
    print(f"Bankroll       : ${BANKROLL:,.2f}")
    print(f"Base Ego       : {live_base_ego:.3f} (Historical Accuracy Weight)\n")
    
    events = []
    if target_slugs:
        # Loop through multiple slugs if provided
        for slug in target_slugs:
            try:
                resp = requests.get(f"https://gamma-api.polymarket.com/events?slug={slug}").json()
                events.extend(resp)
            except Exception as e:
                print(f"API Error fetching slug '{slug}': {e}")
    else:
        try:
            events = requests.get("https://gamma-api.polymarket.com/events?active=true&closed=false&limit=100").json()
        except Exception as e:
            print(f"API Error: {e}")
            return pd.DataFrame()
            
    all_active_markets = []
    for event in events:
        for m in event.get('markets', []):
            if not m.get('active') or m.get('closed') or m.get('umaResolutionStatus') == 'resolved':
                continue
            
            try: pm_mid = float(json.loads(m.get('outcomePrices', '["0.5"]'))[0])
            except: pm_mid = 0.50
            
            # MICROSTRUCTURE: Extreme Odds Filter (Bypassed if user specifically targeted URLs)
            if not target_slugs:
                if pm_mid < EXTREME_ODDS or pm_mid > (1.0 - EXTREME_ODDS):
                    continue
                    
                try:
                    res_dt = datetime.fromisoformat(m.get('endDate').replace('Z', '+00:00'))
                    days_until = (res_dt - datetime.now(timezone.utc)).days
                    if days_until < 0 or days_until > MAX_DAYS:
                        continue 
                except Exception:
                    continue 
            
            market_id = m.get('id')
            
            if mode == "discover" and not target_slugs:
                if market_id in history["predicted"] or market_id in history["skipped"]:
                    continue
            elif mode == "review":
                if sub_mode == "predicted" and market_id not in history["predicted"]: continue
                if sub_mode == "skipped" and market_id not in history["skipped"]: continue
                if sub_mode == "all" and (market_id not in history["predicted"] and market_id not in history["skipped"]): continue

            m['parent_slug'] = event.get('slug', 'unknown-event')
            m['parent_name'] = event.get('title', 'Unknown Event')
            all_active_markets.append(m)

    random.shuffle(all_active_markets) 
    
    if not all_active_markets:
        print("\nNo valid markets found for the selected mode/filters.")
        return pd.DataFrame()

    portfolio_data = []
    session_trades = {}
    cumulative_exposure = 0.0 
    i = 0
    
    while i < len(all_active_markets):
        if cumulative_exposure >= 1.0:
            print("\n[!] Maximum capital exposure reached (100%). Session ending.")
            break
            
        m = all_active_markets[i]
        market_id = m.get('id')
        question = m.get('question', 'Unknown Question')
        event_url = f"https://polymarket.com/event/{m.get('parent_slug')}"
        today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
        event_title = m.get('parent_name')
        
        try:
            res_dt = datetime.fromisoformat(m.get('endDate').replace('Z', '+00:00'))
            exact_date_str = res_dt.strftime("%B %d, %Y")
            days_str = f"{(res_dt - datetime.now(timezone.utc)).days} Days"
        except: exact_date_str, days_str = "Unknown", "Unknown"

        try: pm_mid = float(json.loads(m.get('outcomePrices', '["0.5"]'))[0])
        except: pm_mid = 0.50
        try: pm_bid, pm_ask = float(m.get('bestBid', pm_mid)), float(m.get('bestAsk', pm_mid))
        except: pm_bid, pm_ask = pm_mid, pm_mid
        
        # Bypassed Spread cap if user specifically requested targeted URLs
        if not target_slugs and (pm_ask - pm_bid) > MAX_SPREAD:
            i += 1 
            continue

        print(f"==================================================")
        print(f"Event    : {event_title}")
        print(f"Market   : {question}")
        print(f"Resolves : {exact_date_str} ({days_str})")
        print(f"Exposure : {cumulative_exposure*100:.1f}% deployed")
        
        user_input = input("Enter % value (bounds '16-51' or '42'), 's' (skip), 'r' (redo), 'q' (quit): ").strip()
        
        if user_input.lower() == 'q':
            print("\nSaving and exiting session...")
            break
            
        elif user_input.lower() == 'r':
            if i > 0:
                print("\n[!] Rolling back to previous market...")
                i -= 1 
                prev_market_id = all_active_markets[i].get('id')
                if prev_market_id in session_trades:
                    refund_amount = session_trades.pop(prev_market_id)
                    cumulative_exposure -= (refund_amount / BANKROLL)
                
                history["predicted"].pop(prev_market_id, None)
                history["skipped"].pop(prev_market_id, None)
            else:
                print("\n[!] Cannot redo. This is the first market of the session.\n")
            continue

        elif user_input.lower() == 's': 
            print("Status: Skipped.\n")
            if market_id not in history["predicted"]:
                history["skipped"][market_id] = {"date": today_str}
            i += 1 
            continue
        
        try:
            lower, upper = parse_user_input(user_input)
            volume = float(m.get('volumeNum', 0))
            fee_rate = m.get('feeSchedule', {}).get('rate', 0.05)
            
            action, true_price, adj_kelly, final_alloc, dynamic_ego, edge = calculate_allocation(
                lower, upper, pm_bid, pm_ask, fee_rate, volume, BANKROLL, live_base_ego, KELLY_FRACTION, MAX_VOLUME_IMPACT
            )
            
            history["predicted"][market_id] = {
                "question": question,
                "date": today_str,
                "bounds": f"{(lower*100):.0f}% - {(upper*100):.0f}%",
                "pu": (lower + upper) / 2.0,
                "pm": pm_mid,
                "theoretical_kelly": round(adj_kelly, 4), 
                "allocation": round(final_alloc, 2)
            }
            history["skipped"].pop(market_id, None)
            
            print(f"\n--- MARKET ANALYSIS ---")
            print(f"Market Spread: Bid {pm_bid*100:.1f}% | Ask {pm_ask*100:.1f}% (Spread: {(pm_ask - pm_bid)*100:.1f}%)")
            print(f"User Bounds  : {lower*100:.1f}% to {upper*100:.1f}% (Spread: {(upper-lower)*100:.1f}%)")
            print(f"Dynamic Ego  : {dynamic_ego:.2f} (Base {live_base_ego:.2f})")
            
            if final_alloc > 0:
                cumulative_exposure += final_alloc / BANKROLL
                session_trades[market_id] = final_alloc
                
                print(f"ACTION       : BUY {action} @ {true_price*100:.1f}%")
                print(f"USER EDGE    : {edge*100:.2f}% (After Fees & Spread)")
                print(f"EXPOSURE     : {cumulative_exposure*100:.1f}% of Bankroll Used")
                print(f"FEE RATE     : {fee_rate*100:.1f}%")
                print(f"VOLUME       : ${volume:,.0f} available (Max impact cap: ${volume*MAX_VOLUME_IMPACT:,.2f})")
                print(f"KELLY ALLOC %: {adj_kelly*100:.2f}% of bankroll")
                print(f"FINAL ALLOC %: {final_alloc/BANKROLL*100:.1f}% of bankroll")
                print(f"ALLOCATION   : ${final_alloc:,.2f}")
                print(f"SESSION DATA : {len(session_trades)} predictions | Total Allocated: ${sum(session_trades.values()):,.2f}\n")
                print(f"LINK         : {event_url}\n")
                
                portfolio_data.append({
                    "Question": question[:50] + "..",
                    "Action": action,
                    "Ego": f"{dynamic_ego:.2f}",
                    "Price": f"{true_price*100:.1f}%",
                    "Alloc": f"${final_alloc:,.0f}"
                })
            else:
                if action == "THIN_EDGE": reason = f"Edge below {MIN_EDGE*100}% minimum threshold"
                elif action == "NONE": reason = "Trapped inside bid-ask spread"
                else: reason = "No mathematical edge"
                
                print(f"ACTION       : $0 Allocation ({reason})")
                print(f"USER EDGE    : {edge*100:.2f}% (After Fees & Spread)")
                print(f"EXPOSURE     : {cumulative_exposure*100:.1f}% of Bankroll Used")
                print(f"SESSION DATA : {len(session_trades)} trades | Total Allocated: ${sum(session_trades.values()):,.2f}\n")
                print(f"LINK         : {event_url}\n")
                
            i += 1 
                
        except ValueError: 
            print("\n[!] Error: Invalid input format. Please try again.\n")
            
    save_history(history)
    return pd.DataFrame(portfolio_data)

if __name__ == "__main__":
    print("Select Operation Mode:")
    print(" 1: Discover New Markets (Random)")
    print(" 2: Review 'Predicted' Markets")
    print(" 3: Review 'Skipped' Markets")
    print(" 4: Review All (Predicted + Skipped)")
    print(" 5: Target Specific URLs (Sniper Mode)")
    print(" 6: Complete Reset (Delete History file)")
    
    choice = input("> ").strip()
    
    targets = None
    if choice == "6":
        confirm = input("WARNING: Type 'CONFIRM' to delete all mathematical history and Brier scores: ")
        if confirm == "CONFIRM":
            if os.path.exists(HISTORY_FILE): os.remove(HISTORY_FILE)
            print("Reset complete. Exiting.")
        else:
            print("Reset aborted. Exiting.")
        exit()
    elif choice == "5":
        urls_input = input("Paste Polymarket URLs (comma or space separated):\n> ").replace(',', ' ').split()
        targets = []
        for url in urls_input:
            try:
                # Extracts the slug robustly from standard polymarket.com/event/[slug] formats
                slug = url.split("/event/")[1].split("/")[0].split("?")[0]
                targets.append(slug)
            except IndexError:
                print(f"Invalid URL format skipped: {url}")
        
        if not targets:
            print("No valid URLs parsed. Exiting.")
            exit()
        op_mode, sub_mode = "sniper", "single"
    else:
        mode_map = {
            "1": ("discover", "all"), 
            "2": ("review", "predicted"), 
            "3": ("review", "skipped"),
            "4": ("review", "all")
        }
        op_mode, sub_mode = mode_map.get(choice, ("discover", "all"))
    
    portfolio = run_prediction_session(mode=op_mode, sub_mode=sub_mode, target_slugs=targets)
    
    if not portfolio.empty:
        print("\n--- Final Session Allocations ---")
        pd.set_option('display.max_colwidth', None) 
        print(portfolio.to_string(index=False))
    else:
        print("\nNo allocations made.")

Select Operation Mode:
 1: Discover New Markets (Random)
 2: Review 'Predicted' Markets
 3: Review 'Skipped' Markets
 4: Review All (Predicted + Skipped)
 5: Target Specific URLs (Sniper Mode)
 6: Complete Reset (Delete History file)



--- Starting Session [DISCOVER - ALL] ---
Bankroll       : $42.00
Base Ego       : 0.500 (Historical Accuracy Weight)

Event    : Starmer out by...?
Market   : Starmer out by June 15, 2026?
Resolves : June 15, 2026 (18 Days)
Exposure : 0.0% deployed


KeyboardInterrupt: Interrupted by user

In [52]:
import requests
import json

def analyze_market_layers(offset=0, limit=100):
    print(f"\n[+] FETCHING LAYER DEEP DATA (Offset: {offset} to {offset + limit})...")
    url = f"https://gamma-api.polymarket.com/events?active=true&closed=false&limit={limit}&offset={offset}"
    
    try:
        events = requests.get(url).json()
    except Exception as e:
        print(f"[-] API Error connection: {e}")
        return False

    if not events:
        print("[-] No more active events found in this slice.")
        return False

    print("\n" + "="*115)
    print(f"{'Event Vol':<13} | {'Market Vol':<13} | {'24h Vol':<11} | {'Status':<8} | {'Market Question / Event Title'}")
    print("="*115)

    for event in events:
        # Parent Event Metrics
        event_title = event.get('title', 'Unknown Event')
        event_vol = float(event.get('volume', 0))
        
        # Pull sub-markets inside this container
        markets = event.get('markets', [])
        
        # Print parent summary if it contains no active markets or multiple
        if len(markets) > 1:
            print(f"${event_vol:>11,.0f} | {'[GROUP]':<13} | {'':<11} | {'PARENT'} | --- {event_title} ---")
            
        for m in markets:
            m_question = m.get('question', event_title)
            
            # Handle standard numeric type extraction safely
            m_vol = float(m.get('volumeNum', 0))
            m_24h = float(m.get('volume24hr', 0))
            
            # Safe status determination
            is_active = m.get('active', False)
            is_closed = m.get('closed', False)
            is_resolved = m.get('umaResolutionStatus') == 'resolved'
            
            if is_resolved:
                status = "RESOLVED"
            elif is_closed:
                status = "CLOSED"
            elif is_active:
                status = "ACTIVE"
            else:
                status = "INACTIVE"
                
            # Formatting line printout cleanly without any truncations
            print(f"${event_vol:>11,.0f} | ${m_vol:>11,.0f} | ${m_24h:>9,.0f} | {status:<8} |    ↳ {m_question}")
            
    print("="*115)
    return True

def run_interactive_pager():
    current_offset = 0
    page_size = 100
    
    while True:
        has_data = analyze_market_layers(offset=current_offset, limit=page_size)
        
        if not has_data:
            print("\n[!] End of live market universe stream reached.")
            break
            
        user_choice = input(f"\n[Page {(current_offset//page_size)+1}] Enter 'n' for Next 100, or 'q' to Quit diagnostic: ").strip().lower()
        if user_choice == 'n':
            current_offset += page_size
        elif user_choice == 'q':
            print("\nExiting deep data analyzer.")
            break
        else:
            print("[!] Invalid option. Type 'n' or 'q'.")

if __name__ == "__main__":
    run_interactive_pager()


[+] FETCHING LAYER DEEP DATA (Offset: 0 to 100)...

Event Vol     | Market Vol    | 24h Vol     | Status   | Market Question / Event Title
$ 32,683,392 | [GROUP]       |             | PARENT | --- MicroStrategy sells any Bitcoin by ___ ? ---
$ 32,683,392 | $ 17,976,158 | $        0 | RESOLVED |    ↳ MicroStrategy sells any Bitcoin in 2025?
$ 32,683,392 | $  1,694,512 | $   70,071 | ACTIVE   |    ↳ MicroStrategy sells any Bitcoin by December 31, 2026?
$ 32,683,392 | $  2,710,380 | $        0 | RESOLVED |    ↳ MicroStrategy sells any Bitcoin by March 31, 2026?
$ 32,683,392 | $  4,236,449 | $  155,850 | ACTIVE   |    ↳ MicroStrategy sells any Bitcoin by June 30, 2026?
$ 32,683,392 | $  6,066,161 | $  152,646 | ACTIVE   |    ↳ MicroStrategy sells any Bitcoin by May 31, 2026?
$  1,569,143 | [GROUP]       |             | PARENT | --- Kraken IPO by ___ ? ---
$  1,569,143 | $    494,517 | $        0 | RESOLVED |    ↳ Kraken IPO in 2025?
$  1,569,143 | $    545,639 | $        0 | RESOLVED |   

In [ ]:
import requests
import json
import time

def scan_total_market_universe():
    print("Initializing Global API Scanner...")
    print("Probing pagination limits to find the edge of the market universe...\n")
    
    limit = 100
    offset = 0
    
    total_events = 0
    total_active_markets = 0
    
    # Microstructure Tracking
    spreads = []
    tag_counts = {}
    
    start_time = time.time()
    
    while True:
        url = f"https://gamma-api.polymarket.com/events?active=true&closed=false&limit={limit}&offset={offset}"
        
        try:
            response = requests.get(url)
            events = response.json()
        except Exception as e:
            print(f"[!] Network error at offset {offset}: {e}")
            break
            
        # The API returns an empty list [] when you've passed the last page
        if not events or len(events) == 0:
            print(f"\n[+] END OF UNIVERSE REACHED AT OFFSET: {offset}")
            break
            
        total_events += len(events)
        
        for event in events:
            # Tally Tags/Categories
            tags = event.get('tags', [])
            for tag in tags:
                tag_label = tag.get('label', 'Unknown') if isinstance(tag, dict) else str(tag)
                tag_counts[tag_label] = tag_counts.get(tag_label, 0) + 1
            
            # Dig into markets for liquidity health
            for m in event.get('markets', []):
                if m.get('active') and not m.get('closed') and m.get('umaResolutionStatus') != 'resolved':
                    total_active_markets += 1
                    
                    # Calculate Spreads
                    try:
                        bid = float(m.get('bestBid', 0))
                        ask = float(m.get('bestAsk', 1))
                        # Only measure spread if there's actually a two-sided market
                        if bid > 0 and ask < 1:
                            spread = ask - bid
                            spreads.append(spread)
                    except ValueError:
                        pass
                        
        print(f"Scanned page {(offset//limit)+1} (Offset {offset}...)")
        offset += limit
        
        # Polite sleep to avoid rate limiting
        time.sleep(0.2)

    # --- DATA COMPILATION ---
    elapsed = time.time() - start_time
    avg_spread = (sum(spreads) / len(spreads)) * 100 if spreads else 0
    
    print("\n" + "="*50)
    print("--- GLOBAL POLYMARKET HEALTH REPORT ---")
    print("="*50)
    print(f"Scan Time             : {elapsed:.2f} seconds")
    print(f"Total API Pages       : {offset // limit}")
    print(f"Total Active Events   : {total_events:,}")
    print(f"Total Tradable Markets: {total_active_markets:,}")
    print(f"Global Average Spread : {avg_spread:.2f}%")
    print(f"Healthy Order Books   : {len(spreads):,} (Markets with active bid/ask)")
    
    print("\n--- DOMINANT CATEGORIES (Top 10) ---")
    sorted_tags = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)
    for tag, count in sorted_tags[:10]:
        print(f"{tag:<25} : {count} events")
    print("="*50)

if __name__ == "__main__":
    scan_total_market_universe()

Initializing Global API Scanner...
Probing pagination limits to find the edge of the market universe...

Scanned page 1 (Offset 0...)
Scanned page 2 (Offset 100...)
Scanned page 3 (Offset 200...)
Scanned page 4 (Offset 300...)
Scanned page 5 (Offset 400...)
Scanned page 6 (Offset 500...)
Scanned page 7 (Offset 600...)
Scanned page 8 (Offset 700...)
Scanned page 9 (Offset 800...)
Scanned page 10 (Offset 900...)
Scanned page 11 (Offset 1000...)
Scanned page 12 (Offset 1100...)
Scanned page 13 (Offset 1200...)
Scanned page 14 (Offset 1300...)
Scanned page 15 (Offset 1400...)
Scanned page 16 (Offset 1500...)
Scanned page 17 (Offset 1600...)
Scanned page 18 (Offset 1700...)
Scanned page 19 (Offset 1800...)
Scanned page 20 (Offset 1900...)
Scanned page 21 (Offset 2000...)
Scanned page 22 (Offset 2100...)
Scanned page 23 (Offset 2200...)
Scanned page 24 (Offset 2300...)
Scanned page 25 (Offset 2400...)
Scanned page 26 (Offset 2500...)
Scanned page 27 (Offset 2600...)
Scanned page 28 (Offset 2

### what is "HIDE FROM NEW" category

In [ ]:
import requests

def audit_hidden_markets():
    url = "https://gamma-api.polymarket.com/events?active=true&closed=false&limit=100&offset=0"
    events = requests.get(url).json()
    
    print(f"=== HUNTING 'HIDE FROM NEW' OUTLIERS ===")
    count = 0
    for e in events:
        tags = [t.get('label', '') for t in e.get('tags', []) if isinstance(t, dict)]
        
        if "Hide From New" in tags:
            count += 1
            slug = e.get('slug', 'unknown')
            web_url = f"https://polymarket.com/event/{slug}"
            print(f"\n[{count}] Target Found: {e.get('title')}")
            print(f"    ↳ Slug       : {slug}")
            print(f"    ↳ Total Vol  : ${float(e.get('volume', 0)):,.2f}")
            print(f"    ↳ Web Link   : {web_url}")
            
            # Show a glimpse of the children inside it
            for m in e.get('markets', []):
                print(f"      - Sub-Market ID {m.get('id')}: {m.get('question')} [Status: Active={m.get('active')}]")
                
    if count == 0:
        print("No hidden tags found in the top 100 page. Use a deeper offset to locate unlisted institutional groups.")

if __name__ == "__main__":
    audit_hidden_markets()

=== HUNTING 'HIDE FROM NEW' OUTLIERS ===

[1] Target Found: 2026 FIFA World Cup Winner 
    ↳ Slug       : 2026-fifa-world-cup-winner-595
    ↳ Total Vol  : $1,252,987,627.14
    ↳ Web Link   : https://polymarket.com/event/2026-fifa-world-cup-winner-595
      - Sub-Market ID 558934: Will Spain win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558957: Will New Zealand win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558974: Will Switzerland win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558935: Will England win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558992: Will Team AM win the 2026 FIFA World Cup? [Status: Active=False]
      - Sub-Market ID 558936: Will France win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558961: Will South Korea win the 2026 FIFA World Cup? [Status: Active=True]
      - Sub-Market ID 558977: Will Haiti win the 2026 FIFA World Cup? [Status:

: 